In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import matplotlib.pyplot as plt
import cartopy.feature as cfeature
import cartopy.crs as ccrs
import pandas as pd
import geopandas as gpd
import numpy as np

In [4]:
# dummy dataframe via a csv
# TODO: add dynamic inputs

dat_fp = "/Users/rt582/Library/CloudStorage/OneDrive-UniversityofCambridge/cambridge/phd/blogging/centre-of-mass/test.csv"
dat = gpd.read_file(dat_fp).rename(columns=str.lower)
# create geometry column
dat = gpd.GeoDataFrame(
    dat, geometry=gpd.points_from_xy(dat["lon"], dat["lat"]), crs="EPSG:4326"
)
for col in dat.columns[dat.columns.str.contains("year")]:
    dat[col] = pd.to_datetime(dat[col])
for col in ["lat", "lon"]:
    dat[col] = dat[col].astype(float)


# # trim unnecessary columns. TODO: can do this via read_csv?
bad_cols = dat.columns.str.contains("field")
dat = dat.loc[:, ~bad_cols]
# strip any whitepace from column names and replace spaces with underscores
normalised_colnames = [col.strip().replace(" ", "_") for col in dat.columns]
dat.columns = normalised_colnames
dat.loc[:, "duration"] = (dat["end_year"] - dat["start_year"]).dt.days
dat

,location,lat,lon,start_year,end_year,geometry,duration
0,Oxford,51.5,-1.0,1999-01-01,2004-01-01,POINT (-1 51.5),1826
1,Grosmont,52.0,-2.3,2005-01-01,2017-01-01,POINT (-2.3 52),4383
2,Bristol,51.5,-2.0,2018-01-01,2021-01-01,POINT (-2 51.5),1096
3,London,51.6,0.0,2022-01-01,2022-01-01,POINT (0 51.6),0
4,Cambridge,52.2,0.5,2023-01-01,2026-01-01,POINT (0.5 52.2),1096
5,Wellington,-53.0,180.0,2026-01-01,2027-01-01,POINT (180 -53),365


In [5]:
# Testing: get the time-weighted lat/lon
weighted_lat = ((dat["duration"] * dat["lat"]) / dat["duration"].sum()).sum()
weighted_lon = ((dat["duration"] * dat["lon"]) / dat["duration"].sum()).sum()

print(
    f"Weighted latitude: {weighted_lat:.02f}, weighted longitude: {weighted_lon:.02f}"
)

Weighted latitude: 47.49, weighted longitude: 5.95


In [7]:
import plotly.graph_objects as go
import numpy as np
from PIL import Image

RADIUS = 6400
COLORSCALE = [
    [0.0, "rgb(30, 59, 117)"],
    [0.1, "rgb(46, 68, 21)"],
    [0.2, "rgb(74, 96, 28)"],
    [0.3, "rgb(115,141,90)"],
    [0.4, "rgb(122, 126, 75)"],
    [0.6, "rgb(122, 126, 75)"],
    [0.7, "rgb(141,115,96)"],
    [0.8, "rgb(223, 197, 170)"],
    [0.9, "rgb(237,214,183)"],
    [1.0, "rgb(255, 255, 255)"],
]


def sphere(size, texture):
    N_lat = int(texture.shape[0])
    N_lon = int(texture.shape[1])
    theta = np.linspace(0, 2 * np.pi, N_lat)
    phi = np.linspace(0, np.pi, N_lon)

    # Set up coordinates for points on the sphere
    x0 = size * np.outer(np.cos(theta), np.sin(phi))
    y0 = size * np.outer(np.sin(theta), np.sin(phi))
    z0 = size * np.outer(np.ones(N_lat), np.cos(phi))

    # Set up trace
    return x0, y0, z0


def lon_lat_to_spherical_coords(
    lon: float | np.ndarray, lat: float | np.ndarray, radius: float = RADIUS
) -> tuple(float, float, float):
    """Take latitude and longitude coordinates of a point and calculate the cartesian position"""
    lon_rad = np.deg2rad(lon)
    lat_rad = np.deg2rad(lat)
    x0 = radius * np.cos(lon_rad) * np.cos(lat_rad)
    y0 = radius * np.sin(lon_rad) * np.cos(lat_rad)
    z0 = radius * np.sin(lat_rad)

    return x0, y0, z0


dat["x"], dat["y"], dat["z"] = lon_lat_to_spherical_coords(dat["lon"], dat["lat"])

earth_image_fp = "earth.jpeg"
texture = np.asarray(Image.open(earth_image_fp))[:, :, 0].T
texture = np.roll(texture, texture.shape[0] // 2, axis=0)
x, y, z = sphere(RADIUS, texture)
surf = go.Surface(
    x=x,
    y=y,
    z=z,
    surfacecolor=texture,
    colorscale=COLORSCALE,
    cmin=0,
    cmax=255,
    showscale=False,
)


layout = go.Layout(
    scene=dict(
        aspectratio=dict(x=1, y=1, z=1),
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
    )
)


hovertext = [
    f"{row['location'].capitalize()}<br>Latitude: {row['lat']:.01f} | Longitude: {row['lon']:.01f}"
    for _, row in dat.iterrows()
]

pop_factor = 1.01

fig = go.Figure(data=[surf], layout=layout)
# plot home points
fig.add_trace(
    go.Scatter3d(
        x=dat.x * pop_factor,
        y=dat.y * pop_factor,
        z=dat.z * pop_factor,
        mode="markers",
        marker=dict(color="cyan", opacity=0.5, size=6),
        name="Home point",
        text=hovertext,
        hovertemplate=hovertext,
        # hovertemplate="%{text}<extra></extra>",
        showlegend=False,
    )
)

x_com, y_com, z_com = lon_lat_to_spherical_coords(weighted_lon, weighted_lat)
# pllot C.O.M. point
fig.add_trace(
    go.Scatter3d(
        x=[x_com * pop_factor],
        y=[y_com * pop_factor],
        z=[z_com * pop_factor],
        mode="markers",
        marker=dict(color="red", opacity=1, size=10, symbol="cross"),
        name="Calculated Mass",
        hovertemplate=f"Centre of Mass<br>Latitude: {weighted_lat:.01f} | Longitude: {weighted_lon:.01f}",
    )
)


fig.update_layout(
    scene=dict(
        aspectratio=dict(x=1, y=1, z=1),
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        camera=dict(
            # closer to the globe = more zoomed in
            eye=dict(x=0.5, y=0.8, z=1.0),
            center=dict(x=0, y=0, z=0),
            up=dict(x=0, y=0, z=1),
        ),
    ),
    margin=dict(l=0, r=0, t=0, b=0),
)


fig.update_layout(legend=dict(yanchor="top", y=0.0, xanchor="left", x=0.4))


fig.show()
